# Web Summarizer

In [2]:
import os

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AzureOpenAI

load_dotenv(override=True)

True

## Connect to Azure OpenAI

In [3]:
api_key = os.getenv("GPT_KEY")
endpoint = os.getenv("GPT_ENDPOINT")
api_version = os.getenv("GPT_API_VERSION")
model = os.getenv("GPT_MODEL", "gpt-4o")

missing = [
    name
    for name, value in [
        ("GPT_KEY", api_key),
        ("GPT_ENDPOINT", endpoint),
        ("GPT_API_VERSION", api_version),
    ]
    if not value or value.startswith("your-")
]

if missing:
    print(f"Missing or placeholder values: {', '.join(missing)}")
    print("Copy sample.env to .env and fill in your Azure OpenAI settings.")
else:
    print(f"Azure OpenAI config loaded. Model/deployment: {model}")

client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

Azure OpenAI config loaded. Model/deployment: gpt-4o


## Fetch website text

Simple `requests` + BeautifulSoup scrape. JavaScript-heavy sites and some CDNs may not return useful content.

In [4]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    )
}


def fetch_website_contents(url, max_chars=2000):
    """Return title + body text, truncated to a sensible length."""
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:max_chars]

In [5]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

## Prompts

The model gets a **system** prompt (role and tone) and a **user** prompt (the page text).

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
Use concise language.

"""


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]

In [7]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Summarize a URL

In [8]:
def summarize(url):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(website),
        timeout=120.0,
    )
    return response.choices[0].message.content


def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [9]:
display_summary("https://edwarddonner.com")

This site is Ed Donner’s personal nerd-haven: a CTO/AI-course-guy who writes code, pokes at LLMs for fun, and occasionally makes “very amateur” electronic music while doomscrolling Hacker News.

Highlights:
- He co-founded AI startup Nebula.io, previously sold an AI startup, and did time as a JPMorgan MD, so yes, he’s that kind of overachiever.
- He runs wildly popular Udemy courses on LLMs and AI engineering (almost a million enrollments), with a full curriculum and a recommended course order.
- Recent posts are basically resource dumps for his courses:
  - “AI Coder: Vibe Coder to Agentic Engineer – RESOURCES” (Feb 17, 2026)
  - “AI Builder with n8n – Create Agents and Voice Agents – RESOURCES” (Jan 4, 2026)
  - “AI Engineering MLOps Track – Deploy AI to Production – RESOURCES” (Sept 15, 2025)
  - “Which order to take the AI courses?” (May 28, 2025)

There’s also an “Outsmart” arena for LLMs to duel in diplomacy and deception, because apparently even the models need a PvP mode now.

In [11]:
display_summary("https://en.wikipedia.org/wiki/CERN")

This page is Wikipedia’s love letter to CERN, the giant underground science toy box on the Swiss–French border where physicists smash particles, discover things like the Higgs boson, and accidentally invent the World Wide Web while “just trying to share data.”

Highlights:
- History and founding: A bunch of post-war European countries decide to team up and build the ultimate physics club so their best minds stop emigrating and start colliding.
- Scientific achievements: From fundamental particles to computer science, including the birthplace of the web—yes, your doomscrolling has CERN to thank.
- Giant machines: An entire section on accelerators culminating in the Large Hadron Collider, a 27 km ring whose main job is to yeet protons at each other at almost light speed.
- Money and membership: Lists who pays for this very expensive science hobby and who wants in next.
- Outreach and culture: Public exhibitions, an arts program (because apparently even particle physics needs vibes), and references in popular culture.

No breaking news banner, just a standing reminder that while the rest of us argue online, CERN is literally tearing reality apart to see what falls out.

## With Ollama

Same summarizer, but the chat client points at a local Ollama server instead of Azure.

1. Install and start [Ollama](https://ollama.com)
2. Pull a small, capable model: `ollama pull qwen2.5:3b`
3. Settings come from `.env`: `OLLAMA_BASE_URL` and `OLLAMA_MODEL`

Ollama exposes an OpenAI-compatible API, so we use `OpenAI` with `base_url` and a dummy `api_key`.

In [12]:
from openai import OpenAI

ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
ollama_model = os.getenv("OLLAMA_MODEL", "qwen2.5:3b")

ollama_client = OpenAI(base_url=ollama_base_url, api_key="ollama")

print(f"Ollama base URL: {ollama_base_url}")
print(f"Ollama model: {ollama_model}")

Ollama base URL: http://localhost:11434/v1
Ollama model: qwen2.5:3b


In [13]:
def summarize_with_ollama(url):
    website = fetch_website_contents(url)
    response = ollama_client.chat.completions.create(
        model=ollama_model,
        messages=messages_for(website),
        timeout=120.0,
    )
    return response.choices[0].message.content


def display_summary_ollama(url):
    summary = summarize_with_ollama(url)
    display(Markdown(summary))

In [14]:
display_summary_ollama("https://edwarddonner.com")

Edward Donner's homepage is akin to a tech introvert's overflowing closet: layers upon layers of coding and LLM theory, all with a dash of amateur music production. His main claim to fame? Udemy courses he couldn’t help but lecture about. The avatar adds a quirky touch, though the "About" section barely scratches surface with such bland mentions of recent tech resources. Not exactly a front-runner in personal branding, but with a consistent avatar to keep in sync, who knows?

In [15]:
display_summary_ollama("https://en.wikipedia.org/wiki/CERN")

Snarky summary: A useless page filled with pointless links about a boring physics research center. They seem to think they're cutting-edge just by using fancy L-O-N-G URLs.